# 12a Hazard Index Validation

Transparent benchmark construction, satellite-outcome comparison and paired bootstrap tests.

In [1]:

from pathlib import Path
import hashlib, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

pd.set_option('display.max_columns', 180)
pd.set_option('display.max_rows', 140)
plt.style.use('seaborn-v0_8-whitegrid')
PROJECT_ROOT = Path.cwd() if (Path.cwd() / 'data').exists() else Path.cwd().parent
BASELINE_PATH = PROJECT_ROOT / 'data' / 'processed' / 'dfsi_verified_baseline.csv'
GFD_ANALYSIS_PATH = PROJECT_ROOT / 'outputs' / 'tables' / 'gfd_validation_analysis_sample.csv'
GFD_MATCH_PATH = PROJECT_ROOT / 'outputs' / 'tables' / 'gfd_unique_matches.csv'
GFD_DIR = PROJECT_ROOT / 'data' / 'processed' / 'gfd_external_validation'
TABLE_DIR = PROJECT_ROOT / 'outputs' / 'tables'
FIGURE_DIR = PROJECT_ROOT / 'outputs' / 'figures'
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
for d in [TABLE_DIR, FIGURE_DIR]:
    d.mkdir(parents=True, exist_ok=True)
RANDOM_SEED = 42
BOOTSTRAPS = 2000
PRIMARY = 'cumulative_inundation_area_days_per_km2'

def sha256_file(path):
    h = hashlib.sha256()
    with path.open('rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()
RAW_HASHES_BEFORE = {p.name: sha256_file(p) for p in sorted(RAW_DIR.glob('*.csv'))}

baseline = pd.read_csv(BASELINE_PATH)
gfd = pd.read_csv(GFD_ANALYSIS_PATH)
matches = pd.read_csv(GFD_MATCH_PATH)
event_parts = sorted(GFD_DIR.glob('gfd_india_district_event_metrics_part_*.csv'))
district_events = pd.concat([pd.read_csv(p).assign(source_file=p.name) for p in event_parts], ignore_index=True)
print('Selected inputs')
print('Baseline:', BASELINE_PATH.relative_to(PROJECT_ROOT), len(baseline), baseline.columns.tolist())
print('GFD sample:', GFD_ANALYSIS_PATH.relative_to(PROJECT_ROOT), len(gfd), gfd.columns.tolist())
print('Unique matches:', GFD_MATCH_PATH.relative_to(PROJECT_ROOT), len(matches), matches.columns.tolist())
print('District-event chunks:', len(event_parts), 'rows:', len(district_events), district_events.columns.tolist())

required_columns = ['State_Name','district_name','boundary_shape_id','match_status','flood_event_count','mean_flood_duration','fatalities','injured','corrected_percent_flooded_area','population','published_dfsi','reconstructed_dfsi',PRIMARY,'cumulative_flooded_area_event_fraction','gfd_observed_event_count','cumulative_inundation_area_days_km2_days','cumulative_flooded_area_event_km2','district_area_km2','mean_clear_perc_over_observed_flood_pixels']
missing = [c for c in required_columns if c not in gfd.columns]
assert not missing, f'Missing required GFD analysis columns: {missing}'
assert len(gfd) == 657
assert set(gfd['match_status'].dropna().unique()) <= {'accepted_unique'}

# District-event-derived outcomes.
district_events['boundary_shape_id'] = district_events['boundary_shape_id'].astype(str)
for col in ['flooded_area_km2','flooded_fraction_of_district','mean_inundation_duration','mean_clear_percentage']:
    district_events[col] = pd.to_numeric(district_events[col], errors='coerce').astype('float64')
event_agg = district_events.groupby('boundary_shape_id', dropna=False).agg(
    district_event_positive_rows=('gfd_event_id', 'count'),
    district_event_unique_events=('gfd_event_id', 'nunique'),
    maximum_single_event_flooded_fraction=('flooded_fraction_of_district', 'max'),
    mean_event_inundation_duration_unweighted=('mean_inundation_duration', 'mean')
).reset_index()
analysis = gfd.copy()
analysis['boundary_shape_id'] = analysis['boundary_shape_id'].astype(str)
analysis = analysis.merge(event_agg, on='boundary_shape_id', how='left')
for col in ['district_event_positive_rows','district_event_unique_events']:
    analysis[col] = analysis[col].fillna(0).astype(int)
analysis['maximum_single_event_flooded_fraction'] = analysis['maximum_single_event_flooded_fraction'].fillna(0.0)
analysis['mean_event_inundation_duration'] = np.where(pd.to_numeric(analysis['cumulative_flooded_area_event_km2'], errors='coerce') > 0, pd.to_numeric(analysis['cumulative_inundation_area_days_km2_days'], errors='coerce') / pd.to_numeric(analysis['cumulative_flooded_area_event_km2'], errors='coerce'), np.nan)

# DFSI-derived transformations.
component_cols = ['flood_event_count','mean_flood_duration','fatalities','injured','corrected_percent_flooded_area','population']
for col in component_cols:
    analysis[col] = pd.to_numeric(analysis[col], errors='coerce').astype('float64')
assert analysis[component_cols].notna().all().all()
assert (analysis[['flood_event_count','mean_flood_duration','fatalities','injured','corrected_percent_flooded_area']] >= 0).all().all()
assert (analysis['population'] > 0).all()
analysis['x_events'] = np.log10(1.0 + analysis['flood_event_count'])
analysis['x_duration'] = np.log10(1.0 + analysis['mean_flood_duration'])
analysis['x_flooded_area'] = np.log10(1.0 + analysis['corrected_percent_flooded_area'])
analysis['x_population'] = np.log10(analysis['population'])
analysis['x_fatalities'] = np.log10(1.0 + analysis['fatalities'])
analysis['x_injured'] = np.log10(1.0 + analysis['injured'])
analysis['published_dfsi_benchmark'] = pd.to_numeric(analysis['published_dfsi'], errors='coerce').astype('float64')
analysis['reconstructed_dfsi_benchmark'] = pd.to_numeric(analysis['reconstructed_dfsi'], errors='coerce').astype('float64')
analysis['equal_weight_six_component_log_index'] = analysis[['x_events','x_duration','x_fatalities','x_injured','x_flooded_area','x_population']].mean(axis=1)
analysis['population_component'] = analysis['x_population']
analysis['flooded_area_component'] = analysis['x_flooded_area']
analysis['flood_event_component'] = analysis['x_events']
analysis['duration_component'] = analysis['x_duration']
analysis['hazard_equal_weight_index'] = analysis[['x_events','x_duration','x_flooded_area']].mean(axis=1)
analysis['exposure_score'] = analysis['x_population']
analysis['human_consequence_score'] = analysis[['x_fatalities','x_injured']].mean(axis=1)
analysis.to_csv(TABLE_DIR / 'hazard_index_analysis_dataset.csv', index=False)

input_rows=[]
selected={'state':'State_Name','district':'district_name','boundary_id':'boundary_shape_id','published_dfsi':'published_dfsi','reconstructed_dfsi':'reconstructed_dfsi','flood_event_count':'flood_event_count','mean_flood_duration':'mean_flood_duration','fatalities':'fatalities','injured':'injured','corrected_percent_flooded_area':'corrected_percent_flooded_area','population':'population','primary_satellite_outcome':PRIMARY}
for role,col in selected.items():
    input_rows.append({'source_file':str(GFD_ANALYSIS_PATH.relative_to(PROJECT_ROOT)),'role':role,'selected_column':col,'rows':len(analysis),'missing_count':int(analysis[col].isna().sum()),'unique_count':int(analysis[col].nunique(dropna=True)),'notes':'Selected from existing unique accepted GFD validation analysis sample.'})
pd.DataFrame(input_rows).to_csv(TABLE_DIR / 'hazard_index_input_audit.csv', index=False)

benchmark_specs=[('published_dfsi_benchmark','Published DFSI','six_component_dfsi'),('reconstructed_dfsi_benchmark','Reconstructed DFSI','six_component_dfsi'),('equal_weight_six_component_log_index','Equal-weight six-component log index','six_component_equal'),('population_component','Population component alone','exposure'),('flooded_area_component','Flooded-area component alone','physical_hazard_component'),('flood_event_component','Flood-event component alone','physical_hazard_component'),('duration_component','Duration component alone','physical_hazard_component'),('hazard_equal_weight_index','Equal-weight physical-hazard index','physical_hazard_index')]
outcome_specs=[(PRIMARY,'Cumulative inundation area-days per km2','primary'),('cumulative_flooded_area_event_fraction','Cumulative flooded area-event fraction','secondary'),('gfd_observed_event_count','District-event flood detections','secondary'),('mean_event_inundation_duration','Area-weighted mean event inundation duration','secondary'),('maximum_single_event_flooded_fraction','Maximum single-event flooded fraction','secondary')]

def clean_pair(df,x_col,y_col):
    return df[[x_col,y_col]].replace([np.inf,-np.inf],np.nan).dropna()
def safe_spearman(x,y):
    if len(x)<4 or pd.Series(x).nunique()<2 or pd.Series(y).nunique()<2: return (np.nan,np.nan)
    r=stats.spearmanr(x,y); return float(r.statistic), float(r.pvalue)
def safe_kendall(x,y):
    if len(x)<4 or pd.Series(x).nunique()<2 or pd.Series(y).nunique()<2: return (np.nan,np.nan)
    r=stats.kendalltau(x,y); return float(r.statistic), float(r.pvalue)
def bootstrap_spearman_ci(x,y,n_boot=BOOTSTRAPS,seed=RANDOM_SEED):
    rng=np.random.default_rng(seed); x=np.asarray(x,dtype='float64'); y=np.asarray(y,dtype='float64'); n=len(x); vals=[]
    for _ in range(n_boot):
        idx=rng.integers(0,n,n); r,_=safe_spearman(x[idx],y[idx])
        if np.isfinite(r): vals.append(r)
    if not vals: return np.nan,np.nan
    return tuple(np.percentile(vals,[2.5,97.5]))
def top_overlap(df,index_col,outcome_col):
    dat=clean_pair(df,index_col,outcome_col); n=len(dat)
    if n==0: return n,0,0,np.nan
    k=int(np.ceil(0.10*n)); top_i=set(dat[index_col].rank(method='first',ascending=False).nsmallest(k).index); top_o=set(dat[outcome_col].rank(method='first',ascending=False).nsmallest(k).index); inter=len(top_i&top_o); union=len(top_i|top_o)
    return n,k,inter,inter/union if union else np.nan

rows=[]
for index_col,index_label,group in benchmark_specs:
    for outcome_col,outcome_label,outcome_type in outcome_specs:
        dat=clean_pair(analysis,index_col,outcome_col); x=dat[index_col].to_numpy(); y=dat[outcome_col].to_numpy()
        sp,sp_p=safe_spearman(x,y); ke,ke_p=safe_kendall(x,y); ci_low,ci_high=bootstrap_spearman_ci(x,y); n,top_n,overlap,jaccard=top_overlap(analysis,index_col,outcome_col)
        rows.append({'index_measure':index_col,'index_label':index_label,'benchmark_group':group,'satellite_outcome':outcome_col,'outcome_label':outcome_label,'outcome_type':outcome_type,'usable_districts':len(dat),'spearman_correlation':sp,'spearman_p_value':sp_p,'kendall_correlation':ke,'kendall_p_value':ke_p,'bootstrap_spearman_ci_low':ci_low,'bootstrap_spearman_ci_high':ci_high,'top10_n':top_n,'top10_overlap_count':overlap,'jaccard_similarity':jaccard})
benchmark_results=pd.DataFrame(rows)
benchmark_results.to_csv(TABLE_DIR/'hazard_index_benchmark_results.csv',index=False)
benchmark_results[['index_measure','index_label','benchmark_group','satellite_outcome','usable_districts','top10_n','top10_overlap_count','jaccard_similarity']].to_csv(TABLE_DIR/'hazard_index_top10_comparison.csv',index=False)

qrows=[]
for index_col,index_label,group in benchmark_specs:
    for outcome_col,outcome_label,outcome_type in outcome_specs:
        dat=analysis[[index_col,outcome_col]].replace([np.inf,-np.inf],np.nan).dropna().copy()
        if len(dat)<5 or dat[index_col].nunique()<5: continue
        dat['index_quintile']=pd.qcut(dat[index_col].rank(method='first'),5,labels=['Q1 lowest','Q2','Q3','Q4','Q5 highest'])
        for q,qdf in dat.groupby('index_quintile',observed=False):
            qrows.append({'index_measure':index_col,'index_label':index_label,'satellite_outcome':outcome_col,'outcome_label':outcome_label,'index_quintile':str(q),'districts':len(qdf),'median_satellite_burden':float(qdf[outcome_col].median()),'mean_satellite_burden':float(qdf[outcome_col].mean())})
quintiles=pd.DataFrame(qrows)
quintiles.to_csv(TABLE_DIR/'hazard_index_quintile_results.csv',index=False)

paired_specs=[('hazard_equal_weight_index','published_dfsi_benchmark','Hazard equal weight minus published DFSI'),('hazard_equal_weight_index','reconstructed_dfsi_benchmark','Hazard equal weight minus reconstructed DFSI'),('hazard_equal_weight_index','flooded_area_component','Hazard equal weight minus flooded area alone'),('published_dfsi_benchmark','equal_weight_six_component_log_index','Published DFSI minus equal-weight six-component index'),('published_dfsi_benchmark','population_component','Published DFSI minus population alone')]
def paired_bootstrap(df,a_col,b_col,outcome,n_boot=BOOTSTRAPS,seed=RANDOM_SEED):
    dat=df[[a_col,b_col,outcome]].replace([np.inf,-np.inf],np.nan).dropna(); a=dat[a_col].to_numpy(float); b=dat[b_col].to_numpy(float); y=dat[outcome].to_numpy(float); obs_a,_=safe_spearman(a,y); obs_b,_=safe_spearman(b,y)
    rng=np.random.default_rng(seed); diffs=[]
    for _ in range(n_boot):
        idx=rng.integers(0,len(dat),len(dat)); ra,_=safe_spearman(a[idx],y[idx]); rb,_=safe_spearman(b[idx],y[idx])
        if np.isfinite(ra) and np.isfinite(rb): diffs.append(ra-rb)
    diffs=np.asarray(diffs)
    return {'method_a':a_col,'method_b':b_col,'outcome':outcome,'usable_districts':len(dat),'method_a_observed_spearman':obs_a,'method_b_observed_spearman':obs_b,'observed_difference_spearman':obs_a-obs_b,'bootstrap_median_difference':float(np.nanmedian(diffs)),'bootstrap_ci_low':float(np.nanpercentile(diffs,2.5)),'bootstrap_ci_high':float(np.nanpercentile(diffs,97.5)),'proportion_bootstrap_a_better':float(np.mean(diffs>0)),'bootstrap_samples':len(diffs)}
paired=[]
for a,b,label in paired_specs:
    row=paired_bootstrap(analysis,a,b,PRIMARY); row['comparison']=label; paired.append(row)
paired=pd.DataFrame(paired)
paired.to_csv(TABLE_DIR/'hazard_index_paired_bootstrap_comparisons.csv',index=False)

primary=benchmark_results.query("satellite_outcome == @PRIMARY").sort_values('spearman_correlation',ascending=False)
fig_data=primary.sort_values('spearman_correlation')
fig,ax=plt.subplots(figsize=(9,5.4),constrained_layout=True)
y=np.arange(len(fig_data))
ax.errorbar(fig_data['spearman_correlation'],y,xerr=[fig_data['spearman_correlation']-fig_data['bootstrap_spearman_ci_low'],fig_data['bootstrap_spearman_ci_high']-fig_data['spearman_correlation']],fmt='o',color='#1f4e79',ecolor='#9aa8b3',capsize=3)
ax.axvline(0,color='black',linewidth=0.8); ax.set_yticks(y); ax.set_yticklabels(fig_data['index_label']); ax.set_xlabel('Spearman correlation with cumulative inundation area-days per km2'); ax.set_title('Hazard-index benchmark correlations with satellite-observed inundation'); ax.text(0.01,0.02,'Bootstrap intervals use 2,000 district resamples; n = 657 for the primary outcome.',transform=ax.transAxes,fontsize=8)
fig.savefig(FIGURE_DIR/'hazard_index_benchmark_correlations.png',dpi=220,bbox_inches='tight'); plt.close(fig)

for out in [TABLE_DIR/'hazard_index_analysis_dataset.csv',TABLE_DIR/'hazard_index_input_audit.csv',TABLE_DIR/'hazard_index_benchmark_results.csv',TABLE_DIR/'hazard_index_top10_comparison.csv',TABLE_DIR/'hazard_index_quintile_results.csv',TABLE_DIR/'hazard_index_paired_bootstrap_comparisons.csv',FIGURE_DIR/'hazard_index_benchmark_correlations.png']:
    assert out.exists(), out
assert len(analysis)==657
assert not benchmark_results['spearman_correlation'].isna().all()
assert not paired['observed_difference_spearman'].isna().any()
RAW_HASHES_AFTER={p.name: sha256_file(p) for p in sorted(RAW_DIR.glob('*.csv'))}
assert RAW_HASHES_BEFORE==RAW_HASHES_AFTER, 'data/raw changed during hazard validation.'
print('Saved 12a outputs.')
display(primary[['index_label','usable_districts','spearman_correlation','bootstrap_spearman_ci_low','bootstrap_spearman_ci_high','kendall_correlation','top10_overlap_count','jaccard_similarity']])
display(paired[['comparison','method_a_observed_spearman','method_b_observed_spearman','observed_difference_spearman','bootstrap_ci_low','bootstrap_ci_high','proportion_bootstrap_a_better']])


Selected inputs
Baseline: data/processed/dfsi_verified_baseline.csv 720 ['State_Name', 'district_name', 'available_state_identifier', 'available_state_review_key', 'available_district_identifier', 'available_district_review_key', 'available_district_lgd_codes', 'available_district_lgd_code_count', 'inventory_district_names', 'dfsi_source_row_id', 'impact_source_row_id', 'area_source_row_id', 'flood_event_count', 'mean_flood_duration', 'fatalities', 'injured', 'corrected_percent_flooded_area', 'population', 'published_dfsi', 'reconstructed_dfsi', 'reconstructed_dfsi_product_space', 'log_product_agreement_abs_diff', 'published_rank', 'reconstructed_rank', 'numerical_difference', 'absolute_difference', 'rank_difference', 'absolute_rank_difference', 'completeness_status', 'join_status']
GFD sample: outputs/tables/gfd_validation_analysis_sample.csv 657 ['dfsi_record_key_int', 'State_Name', 'district_name', 'available_state_identifier', 'available_state_review_key', 'available_district_ident

Saved 12a outputs.


,index_label,usable_districts,spearman_correlation,bootstrap_spearman_ci_low,bootstrap_spearman_ci_high,kendall_correlation,top10_overlap_count,jaccard_similarity
35,Equal-weight physical-hazard index,657,0.452920,0.383979,0.520003,0.323931,43,0.483146
15,Population component alone,657,0.394382,0.319629,0.460494,0.273118,13,0.109244
10,Equal-weight six-component log index,657,0.387765,0.311679,0.459853,0.270528,28,0.269231
30,Duration component alone,657,0.384091,0.319258,0.443147,0.261753,9,0.073171
20,Flooded-area component alone,657,0.374484,0.292589,0.448308,0.255994,39,0.419355
5,Reconstructed DFSI,657,0.372106,0.296356,0.444124,0.259155,25,0.233645
0,Published DFSI,657,0.357479,0.277381,0.429997,0.248589,25,0.233645
25,Flood-event component alone,657,0.298675,0.220937,0.373215,0.210804,20,0.178571


,comparison,method_a_observed_spearman,method_b_observed_spearman,observed_difference_spearman,bootstrap_ci_low,bootstrap_ci_high,proportion_bootstrap_a_better
0,Hazard equal weight minus published DFSI,0.452920,0.357479,0.095441,0.043603,0.150492,1.000
1,Hazard equal weight minus reconstructed DFSI,0.452920,0.372106,0.080814,0.032952,0.130385,1.000
2,Hazard equal weight minus flooded area alone,0.452920,0.374484,0.078436,0.011575,0.147363,0.984
3,Published DFSI minus equal-weight six-componen...,0.357479,0.387765,-0.030286,-0.052898,-0.008264,0.003
4,Published DFSI minus population alone,0.357479,0.394382,-0.036904,-0.099776,0.029440,0.152
